# HydroSeason 0.1.1 improvements

This self-contained demonstration needs only the core `hydroseason` install, makes no network calls, and takes a few seconds. It uses the committed 21-year Fitzroy River (WA) extent fixture plus small synthetic examples to show the 0.1.1 timing, reporting, AOI, and multi-AOI contracts.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import FileLink, display

from hydroseason import analyze_catchment, generate_catchment_report, run_hydroseason
from hydroseason._aoi_context import AOIContext, build_aoi_context

# Work whether the kernel starts in the repository root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'case_studies').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CASE_STUDY_CSV = PROJECT_ROOT / 'case_studies/data/extent/fitzroy_river_wa_30m.csv'
OUTPUT_DIR = PROJECT_ROOT / 'notebooks/output/05_0_1_improvements'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert CASE_STUDY_CSV.exists(), CASE_STUDY_CSV


## Real case: deterministic timing evidence

Fitzroy is analysed from its committed monthly CSV. The deterministic controls make the bootstrap interval and discrete-null Kuiper simulation reproducible; `show_map=False` keeps this real-case run completely offline.

In [ ]:
fitzroy = run_hydroseason(
    CASE_STUDY_CSV,
    output_dir=OUTPUT_DIR / 'fitzroy',
    aoi_name='Fitzroy River (WA)',
    analysis_options={'n_bootstrap': 999, 'random_state': 0},
    show_map=False,
)

timing = fitzroy.analysis.regime
summary = pd.DataFrame([
    {
        'regime': timing.regime,
        'route': fitzroy.analysis.route,
        'SNR': timing.amplitude_snr,
        'peak R': timing.peak_timing_concentration,
        'peak 95% CI': (timing.peak_timing_concentration_ci_low, timing.peak_timing_concentration_ci_high),
        'peak Kuiper p': timing.peak_timing_uniformity_p,
        'peak IQR (months)': timing.peak_phase_iqr_months,
        'trough R': timing.trough_timing_concentration,
        'trough 95% CI': (timing.trough_timing_concentration_ci_low, timing.trough_timing_concentration_ci_high),
        'trough Kuiper p': timing.trough_timing_uniformity_p,
        'trough IQR (months)': timing.trough_phase_iqr_months,
        'n_timing_years': timing.n_timing_years,
    }
])
summary.round(3)


The report is a local, self-contained HTML summary. Its location comes from the returned artifact rather than a documentation URL.

In [ ]:
html_path = fitzroy.artifacts.html
print(f'Open the generated HTML summary: {html_path}')
display(FileLink(html_path))


## CSV bundle compatibility

Version 0.1.1 does not substantially change the monthly, hydrological-year, wet-event, or low-spell CSV schemas. Timing evidence is presented in the analysis summary and HTML; the route and its reasons are what changed. This check reveals any added or removed columns instead of concealing a schema change.

In [ ]:
expected_schemas = {
    'monthly': ['date', 'extent_pct', 'invalid_pct', 'max_invalid_pct', 'baseline_extent_pct', 'usable_month', 'quality_state', 'hy_year', 'phase', 'phase_status', 'is_hy_peak', 'is_hy_mid_dry', 'is_hy_trough', 'in_wet_event', 'wet_event_id', 'in_low_spell', 'low_spell_id', 'regime', 'route'],
    'hydrological-year': ['catchment', 'hy_year', 'start_date', 'end_date', 'peak_date', 'mid_dry_date', 'trough_date', 'peak_extent_pct', 'mid_dry_extent_pct', 'trough_extent_pct', 'peak_invalid_pct', 'mid_dry_invalid_pct', 'trough_invalid_pct', 'drawdown_pct', 'confidence', 'status', 'boundary_status', 'boundary_basis', 'regime', 'route'],
    'wet-event': ['event_id', 'start_date', 'end_date', 'duration_months', 'baseline_extent_pct', 'peak_date', 'peak_extent_pct', 'mean_extent_pct', 'magnitude_pp_months'],
    'low-spell': ['low_spell_id', 'start_date', 'end_date', 'duration_months', 'baseline_extent_pct', 'min_extent_pct'],
}
bundle_paths = {
    'monthly': fitzroy.artifacts.monthly_csv,
    'hydrological-year': fitzroy.artifacts.hydro_years_csv,
    'wet-event': fitzroy.artifacts.wet_event_csv,
    'low-spell': fitzroy.artifacts.low_spells_csv,
}
schema_check = []
for name, path in bundle_paths.items():
    current = list(pd.read_csv(path, nrows=0).columns)
    expected = expected_schemas[name]
    schema_check.append({
        'CSV': name,
        'current columns': ', '.join(current),
        'added': ', '.join(column for column in current if column not in expected) or 'none',
        'removed': ', '.join(column for column in expected if column not in current) or 'none',
    })
pd.DataFrame(schema_check)


## Short records: a low-power guard

A seven-year record can be amplitude-strong but have too little timing evidence to reject the discrete-uniform null. HydroSeason retains that classification as `marginal` rather than mislabelling it `aseasonal`; the caveat makes the low power explicit.

In [ ]:
dates = pd.date_range('2010-01-01', periods=12 * 7, freq='MS')
values = [8.0] * (12 * 7)
for year, month in enumerate([1, 3, 5, 7, 9, 11, 1]):
    values[12 * year + month - 1] = 8.1
short_record = pd.DataFrame({'extent_pct': values, 'invalid_pct': 0.0}, index=dates)
short_timing = analyze_catchment(short_record, n_bootstrap=999, random_state=0).regime
assert short_timing.regime == 'marginal'
assert short_timing.n_timing_years == 7
pd.DataFrame([{
    'regime': short_timing.regime,
    'n_timing_years': short_timing.n_timing_years,
    'SNR': short_timing.amplitude_snr,
    'low-power caveat': ' '.join(short_timing.caveats),
}])


## Interval sensitivity is simulation uncertainty, not a new statistic

The concentration point estimate is calculated from the same annual timings. Changing the number of bootstrap resamples with the same seed can move interval endpoints slightly: that is Monte Carlo interval-estimation variation, not a different R statistic.

In [ ]:
fitzroy_extent = pd.read_csv(CASE_STUDY_CSV, parse_dates=['date']).set_index('date')
sensitivity = []
for n_bootstrap in (199, 999):
    assessment = analyze_catchment(
        fitzroy_extent, n_bootstrap=n_bootstrap, random_state=0
    ).regime
    sensitivity.append({
        'bootstrap resamples': n_bootstrap,
        'peak R': assessment.peak_timing_concentration,
        'peak CI low': assessment.peak_timing_concentration_ci_low,
        'peak CI high': assessment.peak_timing_concentration_ci_high,
    })
pd.DataFrame(sensitivity).round(4)


## AOI-aware reports without acquisition

This uses a small valid synthetic GeoDataFrame when the optional geospatial packages are installed, with a dependency-light `AOIContext` fallback so the notebook still needs only the core install. No tiles are fetched: the report embeds the boundary and Leaflet runtime, while OSM basemap tiles are only requested later if a viewer opens the map online.

In [ ]:
try:
    import geopandas as gpd
    from shapely.geometry import Polygon

    synthetic_gdf = gpd.GeoDataFrame(
        geometry=[Polygon([(120.0, -18.0), (120.1, -18.0), (120.1, -18.1), (120.0, -18.1)])],
        crs='EPSG:4326',
    )
    synthetic_aoi = build_aoi_context(synthetic_gdf, display_name='Synthetic offline AOI')
except ImportError:
    synthetic_aoi = AOIContext(
        geojson='{"type":"FeatureCollection","features":[{"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[120.0,-18.0],[120.1,-18.0],[120.1,-18.1],[120.0,-18.1],[120.0,-18.0]]]},"properties":{}}]}',
        bounds_wgs84=(120.0, -18.1, 120.1, -18.0),
        display_name='Synthetic offline AOI',
        feature_count=1,
    )
aoi_report = generate_catchment_report(
    fitzroy.extent,
    OUTPUT_DIR / 'synthetic-aoi-report',
    name='Synthetic offline AOI',
    analysis=fitzroy.analysis,
    aoi_context=synthetic_aoi,
)
aoi_html = aoi_report.html.read_text(encoding='utf-8')
assert 'id="aoi-context"' in aoi_html
assert 'peak timing concentration' in aoi_html
print(f'Open the AOI-aware offline report: {aoi_report.html}')
display(FileLink(aoi_report.html))


## Multi-AOI means one result per source row

`run_hydroseason_many` is the DEA/STAC batch API, but this notebook deliberately does not make a remote call. The synthetic contract table below illustrates its row-preserving result model: separate input rows remain separate outputs, while one MultiPolygon stored in one row remains one AOI.

In [ ]:
multi_aoi_contract = pd.DataFrame([
    {'source row': 0, 'source geometry': 'Polygon', 'output directory': 'batch/fitzroy', 'outcome position': 0},
    {'source row': 1, 'source geometry': 'Polygon', 'output directory': 'batch/gilbert', 'outcome position': 1},
    {'source row': 2, 'source geometry': 'MultiPolygon', 'output directory': 'batch/one-multipolygon', 'outcome position': 2},
])
multi_aoi_contract


## What changed in 0.1.1

| Area | Change |
| --- | --- |
| Seasonality | Peak/trough circular concentration, bootstrap intervals, and discrete-null Kuiper evidence replace the old fixed IQR decision gate. |
| Routing | Trough timing now controls whether per-year boundaries are defensible. |
| Reports | HTML summaries expose timing evidence and can include an embedded AOI boundary. |
| Batch work | `run_hydroseason_many` preserves source rows and applies memory-aware scheduling for DEA/STAC work. |

Limitations: this notebook did not run a live DEA/STAC batch; map basemap tiles remain a view-time OpenStreetMap dependency; and Fitzroy has 21 timing observations, so it retains the fewer-than-30-observations caution.